# Sen1Floods11: corrected data preparation and baseline training

This notebook uses the dataset's **252/89/90 IID split** and keeps the **15 Bolivia tiles** as a separate event-level generalization holdout. It validates that the Kaggle 8-channel stack matches the original source, then trains a baseline with the verified channel order.

**Verified channel order (0-based):** optical `OPT=[0,1,2,3]` = S2 Red, Green, Blue, NIR; SAR `SAR=[4,5]` = S1 VV, VH; `DEM=[6]` is retained as the auxiliary seventh input; raw channel 7 (8th band) is excluded. Note: source comparison did not conclusively identify the exact identities of the auxiliary channels.

Run top to bottom in Kaggle with GPU enabled and Internet enabled for retrieving original split lists and source samples. The notebook will stop if required checks fail. Do not use the Kaggle copy's `312/67/67` split files as the IID benchmark.

In [1]:
import os, re, io, json, math, random, time, requests
import numpy as np, pandas as pd, rasterio, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from scipy.spatial import cKDTree

DATASET_DIR = '/kaggle/input/datasets/oindrieelmondal/sen1floods11-8-channel-remote-sensing-dataset/dataset/Sen1Floods11_8Channel'
IMAGE_DIR = os.path.join(DATASET_DIR, 'image')
LABEL_DIR = os.path.join(DATASET_DIR, 'label')
WORK_DIR = '/kaggle/working/corrected_run'
os.makedirs(WORK_DIR, exist_ok=True)

# Single source of truth for all modality routing.
OPT = [0, 1, 2, 3]
SAR = [4, 5]
DEM = [6]
MODEL_RAW_IDX = OPT + SAR + DEM
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
assert len(OPT) == 4 and len(SAR) == 2 and len(DEM) == 1

Device: cuda


## 1. Audit the local tiles and channel values

In [2]:
image_files = sorted([p for p in os.listdir(IMAGE_DIR) if p.endswith('_image.tif')])
label_files = sorted([p for p in os.listdir(LABEL_DIR) if p.endswith('_label.tif')])
image_ids = {p.replace('_image.tif','') for p in image_files}
label_ids = {p.replace('_label.tif','') for p in label_files}
assert image_ids == label_ids, f'Image/label ID mismatch: {len(image_ids-label_ids)} images missing labels, {len(label_ids-image_ids)} labels missing images'
assert len(image_ids) == 446, f'Expected 446 hand-labeled tiles, found {len(image_ids)}'

# Audit all tiles, not only the first sample.
raw_nan_by_channel = np.zeros(8, dtype=np.int64)
raw_total_by_channel = np.zeros(8, dtype=np.int64)
raw_sums = np.zeros(8, dtype=np.float64)
raw_counts = np.zeros(8, dtype=np.int64)
for tid in sorted(image_ids):
    with rasterio.open(os.path.join(IMAGE_DIR, f'{tid}_image.tif')) as src:
        x = src.read().astype(np.float32)
    assert x.shape[0] == 8 and x.shape[1:] == (512,512), (tid, x.shape)
    for c in range(8):
        good = np.isfinite(x[c])
        raw_nan_by_channel[c] += (~good).sum()
        raw_total_by_channel[c] += x[c].size
        raw_sums[c] += x[c][good].sum(dtype=np.float64)
        raw_counts[c] += good.sum()
means = raw_sums / np.maximum(raw_counts,1)
print('Raw band means:', np.round(means,3).tolist())
print('NaN pixel counts by raw channel:', raw_nan_by_channel.tolist())
# Based on source-verified order: optical bands are nonnegative reflectance; SAR is dB-like.
assert np.nanmin(means[OPT]) >= 0 and float(np.mean(means[OPT])) >= 1000, 'Optical range/mean check failed; inspect channel order.'
assert float(np.mean(means[SAR])) < 0, 'SAR mean is not negative dB-like; inspect channel order.'
assert int(raw_nan_by_channel[OPT].sum()) == 0, 'Unexpected NaN in optical bands; investigate before training.'
assert int(raw_nan_by_channel[DEM].sum()) == 0, 'Unexpected NaN in DEM band; investigate before training.'
assert int(raw_nan_by_channel[7]) == 0, 'Unexpected NaN in excluded band; investigate.'
assert int(raw_nan_by_channel[SAR].sum()) > 0, 'Expected observed NaNs in SAR bands were not found; verify dataset version.'
print('PASS: first-batch-compatible band checks: OPT nonnegative/high reflectance; SAR negative dB-like; NaNs only in SAR.')

with rasterio.open(os.path.join(LABEL_DIR, sorted(label_ids)[0]+'_label.tif')) as src:
    print('Example label classes:', np.unique(src.read(1)).tolist(), '(expected -1 invalid, 0 land, 1 water)')

Raw band means: [1195.157, 1344.43, 1379.89, 2578.411, -10.435, -17.308, 155.355, 18.131]
NaN pixel counts by raw channel: [0, 0, 0, 0, 2511705, 2511705, 0, 0]
PASS: first-batch-compatible band checks: OPT nonnegative/high reflectance; SAR negative dB-like; NaNs only in SAR.
Example label classes: [-1, 0, 1] (expected -1 invalid, 0 land, 1 water)


## 2. Load the original IID split and recover event names

The Kaggle repackaging's local split files may be 312/67/67. This cell fetches the original release split CSVs from the public Sen1Floods11 v1.1 bucket, extracts IDs that exist in the local stack, and verifies the expected partition sizes. It also retrieves original filename prefixes to identify the event without guessing from bounding boxes.

In [4]:
BUCKET='sen1floods11'
API='https://storage.googleapis.com/storage/v1/b/'+BUCKET+'/o'

def list_objects(prefix):
    names=[]; token=None
    while True:
        params={'prefix':prefix,'maxResults':1000,'fields':'items(name),nextPageToken'}
        if token: params['pageToken']=token
        r=requests.get(API,params=params,timeout=60); r.raise_for_status(); j=r.json()
        names.extend(x['name'] for x in j.get('items',[])); token=j.get('nextPageToken')
        if not token: break
    return names

def get_object(name):
    url=f'https://storage.googleapis.com/{BUCKET}/{name}'
    r=requests.get(url,timeout=120); r.raise_for_status(); return r.content

split_prefix='v1.1/splits/flood_handlabeled/'
split_objects=list_objects(split_prefix)
print('Original split objects:', split_objects)
assert split_objects, 'Could not list original split CSVs. Enable Kaggle Internet and rerun.'

def ids_from_csv_object(name):
    raw=get_object(name)
    df=pd.read_csv(io.BytesIO(raw),header=None,dtype=str,keep_default_na=False)
    found=set()
    for value in df.astype(str).to_numpy().ravel():
        for token in re.findall(r"(?<!\d)(\d+)(?!\d)", value):
            if token in image_ids: found.add(token)
    return found

def split_name(name):
    n=name.lower()
    if 'train' in n: return 'train'
    if 'val' in n or 'valid' in n: return 'validation'
    if 'test' in n: return 'test'
    return None

official={}
for obj in split_objects:
    # Bolivia is an extra holdout, never part of the IID train/val/test lists.
    if 'bolivia' in obj.lower(): continue
    kind=split_name(obj)
    if kind:
        ids=ids_from_csv_object(obj)
        if ids: official[kind]=ids
assert set(official)=={'train','validation','test'}, f'Could not identify all original split CSVs: {list(official)}'
expected={'train':252,'validation':89,'test':90}
for k,n in expected.items(): assert len(official[k])==n, f'{k} count {len(official[k])}, expected {n}; inspect source CSV parsing.'
assert not (official['train']&official['validation'] or official['train']&official['test'] or official['validation']&official['test']), 'Original IID split IDs overlap.'
iid_ids=set.union(*official.values())
assert len(iid_ids)==431, f'Expected 431 IID IDs, got {len(iid_ids)}'

# Recover event prefix from original LabelHand filenames, matched by unique numeric tile ID.
label_prefix='v1.1/data/flood_events/HandLabeled/LabelHand/'
label_objects=list_objects(label_prefix)
event_by_id={}
for name in label_objects:
    m=re.search(r'/([^/]+?)_(\d+)_LabelHand\.tif$',name)
    if m and m.group(2) in image_ids:
        event_by_id[m.group(2)]=m.group(1)
missing_event=image_ids-set(event_by_id)
assert not missing_event, f'No original event prefix for {len(missing_event)} local tiles; examples {list(missing_event)[:5]}'
assert 'Unknown' not in set(event_by_id.values())
bolivia={tid for tid,event in event_by_id.items() if event.lower()=='bolivia'}
assert len(bolivia)==15, f'Expected 15 Bolivia OOD tiles, found {len(bolivia)}'
assert iid_ids.isdisjoint(bolivia), 'Bolivia must be outside the IID train/validation/test split.'
assert iid_ids|bolivia == image_ids, f'Splits and Bolivia do not cover all tiles; missing={len(image_ids-(iid_ids|bolivia))}, extra={len((iid_ids|bolivia)-image_ids)}'
print('PASS split sizes:',{k:len(v) for k,v in official.items()},'Bolivia OOD:',len(bolivia))

Original split objects: ['v1.1/splits/flood_handlabeled/flood_bolivia_data.csv', 'v1.1/splits/flood_handlabeled/flood_test_data.csv', 'v1.1/splits/flood_handlabeled/flood_train_data.csv', 'v1.1/splits/flood_handlabeled/flood_valid_data.csv']
PASS split sizes: {'test': 90, 'train': 252, 'validation': 89} Bolivia OOD: 15


In [5]:
# Save the resolved IDs and event-stratified cross-tab for reproducibility.
rows=[]
for split,ids in official.items():
    rows += [{'id':tid,'event':event_by_id[tid],'split':split} for tid in sorted(ids)]
rows += [{'id':tid,'event':event_by_id[tid],'split':'Bolivia OOD'} for tid in sorted(bolivia)]
split_df=pd.DataFrame(rows)
assert split_df['id'].nunique()==446 and set(split_df.id)==image_ids
print(pd.crosstab(split_df['event'],split_df['split'],margins=True))
split_df.to_csv(os.path.join(WORK_DIR,'verified_split_manifest.csv'),index=False)

# Confirm benchmark IDs match the source list exactly; never substitute the local 312/67/67 list.
for k,ids in official.items():
    print(k, 'first IDs:', sorted(ids)[:5])

# Tile-centre nearest-neighbour distances: descriptive overlap check, not proof of leakage.
def tile_center(tid):
    with rasterio.open(os.path.join(IMAGE_DIR,f'{tid}_image.tif')) as src:
        b=src.bounds
    return ((b.left+b.right)/2,(b.bottom+b.top)/2)
train_xy=np.array([tile_center(t) for t in official['train']])
test_xy=np.array([tile_center(t) for t in official['test']])
# Haversine distance in km from lon/lat centers.
def haversine_km(a,b):
    lon1,lat1=np.radians(a[:,0]),np.radians(a[:,1]); lon2,lat2=np.radians(b[:,0]),np.radians(b[:,1])
    dlon=lon2[None,:]-lon1[:,None]; dlat=lat2[None,:]-lat1[:,None]
    h=np.sin(dlat/2)**2+np.cos(lat1[:,None])*np.cos(lat2[None,:])*np.sin(dlon/2)**2
    return 6371*2*np.arcsin(np.sqrt(np.clip(h,0,1)))
nearest=haversine_km(test_xy,train_xy).min(axis=1)
print('IID test-to-train nearest centre km: min/median/max =',np.round([nearest.min(),np.median(nearest),nearest.max()],2).tolist())

split      Bolivia OOD  test  train  validation  All
event                                               
Bolivia             15     0      0           0   15
Ghana                0    11     31          11   53
India                0    14     40          14   68
Mekong               0     6     18           6   30
Nigeria              0     4     10           4   18
Pakistan             0     6     16           6   28
Paraguay             0    14     39          14   67
Somalia              0     6     15           5   26
Spain                0     6     18           6   30
Sri-Lanka            0     9     24           9   42
USA                  0    14     41          14   69
All                 15    90    252          89  446
test first IDs: ['1018327', '1029191', '1049586', '1049830', '1078550']
train first IDs: ['1010394', '1017769', '1029042', '103272', '103447']
validation first IDs: ['1018317', '101973', '1019808', '1027214', '1033830']
IID test-to-train nearest centre km: m

## 3. Verify the first six raw channels against original source chips

This repeats the source comparison across five events. It downloads only the selected original chips and asserts Pearson correlation ≥0.995 for each expected band match. If internet is unavailable or a source file cannot be matched, training stops rather than silently trusting old hand-written display labels.

In [6]:
from scipy.stats import pearsonr
S1_PREFIX='v1.1/data/flood_events/HandLabeled/S1Hand/'
S2_PREFIX='v1.1/data/flood_events/HandLabeled/S2Hand/'
s1_objects=list_objects(S1_PREFIX); s2_objects=list_objects(S2_PREFIX)
def object_by_id(objects,tid,layer):
    hits=[n for n in objects if re.search(rf'_{re.escape(tid)}_{layer}\.tif$',n)]
    if len(hits)!=1: raise RuntimeError(f'Expected one {layer} source for {tid}; got {hits[:3]}')
    return hits[0]
# Select one tile from each of five distinct non-Bolivia events.
event_candidates={}
for tid in sorted(iid_ids): event_candidates.setdefault(event_by_id[tid],tid)
sample_ids=[event_candidates[e] for e in sorted(event_candidates) if e.lower()!='bolivia'][:5]
assert len(sample_ids)==5, 'Need five different IID events for channel confirmation.'
# Kaggle raw channels: 0..3 map to S2 B4,B3,B2,B8; 4..5 map to S1 VV,VH.
s2_band_indices=[3,2,1,7]
mapping=[('S2',s2_band_indices[0]),('S2',s2_band_indices[1]),('S2',s2_band_indices[2]),('S2',s2_band_indices[3]),('S1',0),('S1',1)]
rhos={i:[] for i in range(6)}
for tid in sample_ids:
    with rasterio.open(os.path.join(IMAGE_DIR,f'{tid}_image.tif')) as s: local=s.read().astype(np.float32)
    s1name=object_by_id(s1_objects,tid,'S1Hand'); s2name=object_by_id(s2_objects,tid,'S2Hand')
    s1bytes=get_object(s1name); s2bytes=get_object(s2name)
    with rasterio.MemoryFile(s1bytes) as mem:
        with mem.open() as s: s1=s.read().astype(np.float32)
    with rasterio.MemoryFile(s2bytes) as mem:
        with mem.open() as s: s2=s.read().astype(np.float32)
    originals=[s2[3],s2[2],s2[1],s2[7],s1[0],s1[1]]
    for c,ref in enumerate(originals):
        a=local[c].ravel(); b=ref.ravel(); good=np.isfinite(a)&np.isfinite(b)
        rho=float(pearsonr(a[good],b[good]).statistic)
        rhos[c].append(rho)
        print(event_by_id[tid],tid,'raw channel',c,'rho',round(rho,6))
for c,vals in rhos.items(): assert min(vals)>=0.995, f'Raw channel {c} source match failed: {vals}'
print('PASS: five-event source check; each raw band match has rho >= 0.995.')

Ghana 103272 raw channel 0 rho 1.0
Ghana 103272 raw channel 1 rho 0.999998
Ghana 103272 raw channel 2 rho 1.0
Ghana 103272 raw channel 3 rho 1.0
Ghana 103272 raw channel 4 rho 1.0
Ghana 103272 raw channel 5 rho 0.999999
India 1017769 raw channel 0 rho 1.0
India 1017769 raw channel 1 rho 1.0
India 1017769 raw channel 2 rho 1.0
India 1017769 raw channel 3 rho 1.0
India 1017769 raw channel 4 rho 1.0
India 1017769 raw channel 5 rho 1.0
Mekong 1111068 raw channel 0 rho 1.0
Mekong 1111068 raw channel 1 rho 1.0
Mekong 1111068 raw channel 2 rho 1.0
Mekong 1111068 raw channel 3 rho 1.0
Mekong 1111068 raw channel 4 rho 1.0
Mekong 1111068 raw channel 5 rho 1.0
Nigeria 1095404 raw channel 0 rho 1.0
Nigeria 1095404 raw channel 1 rho 1.0
Nigeria 1095404 raw channel 2 rho 0.999999
Nigeria 1095404 raw channel 3 rho 1.0
Nigeria 1095404 raw channel 4 rho 1.0
Nigeria 1095404 raw channel 5 rho 1.0
Pakistan 1027214 raw channel 0 rho 1.0
Pakistan 1027214 raw channel 1 rho 1.0
Pakistan 1027214 raw channel 2 

## Toy routing assertion: optical ablation changes OPT only


In [7]:
# x has seven model channels in verified order: optical, SAR, DEM.
x=torch.arange(1*7*2*2,dtype=torch.float32).reshape(1,7,2,2)
def zero_group(batch, indices):
    y=batch.clone(); y[:,indices,:,:]=0; return y
x_opt = zero_group(x, OPT)

expected_x = x.clone()
expected_x[:, OPT, :, :] = 0

assert torch.equal(x_opt, expected_x), "Optical ablation produced unexpected values."
assert torch.equal(x_opt[:, SAR + DEM, :, :], x[:, SAR + DEM, :, :])

print("PASS: optical channels are zeroed; SAR and DEM are preserved.")

PASS: optical channels are zeroed; SAR and DEM are preserved.


## Train five fresh seeds and select checkpoints on validation harmonic mean

In [8]:
# This reuses the corrected verified dataset definitions above.
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

class FloodDataset(Dataset):
    def __init__(self, ids, means, stds): self.ids=sorted(ids); self.means=means[:,None,None]; self.stds=stds[:,None,None]
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        tid=self.ids[i]
        with rasterio.open(os.path.join(IMAGE_DIR,f'{tid}_image.tif')) as s: raw=s.read().astype(np.float32)
        image=raw[MODEL_RAW_IDX].copy()
        with rasterio.open(os.path.join(LABEL_DIR,f'{tid}_label.tif')) as s: label=s.read(1).astype(np.int64)
        ignore=(label<0)|(label>1)
        for c in range(7):
            bad=~np.isfinite(image[c])
            if bad.any():
                assert c in SAR, f'Unexpected NaN in model channel {c}'
                image[c,bad]=0.; ignore |= bad
            good=~bad; image[c,good]=(image[c,good]-self.means[c,0,0])/self.stds[c,0,0]
        label[ignore]=255
        return torch.from_numpy(image),torch.from_numpy(label)

sums=np.zeros(7,np.float64); sq=np.zeros(7,np.float64); counts=np.zeros(7,np.float64)
for tid in official['train']:
    with rasterio.open(os.path.join(IMAGE_DIR,f'{tid}_image.tif')) as s: raw=s.read().astype(np.float32)
    for c,band in enumerate(raw[MODEL_RAW_IDX]):
        vals=band[np.isfinite(band)].astype(np.float64); sums[c]+=vals.sum(); sq[c]+=(vals**2).sum(); counts[c]+=len(vals)
means=(sums/np.maximum(counts,1)).astype(np.float32); stds=np.sqrt(np.maximum(sq/np.maximum(counts,1)-means**2,1e-8)).astype(np.float32)
train_ds=FloodDataset(official['train'],means,stds)
valid_loader=DataLoader(FloodDataset(official['validation'],means,stds),batch_size=4,shuffle=False,num_workers=2)
test_loader=DataLoader(FloodDataset(official['test'],means,stds),batch_size=4,shuffle=False,num_workers=2)
cal_loader=DataLoader(train_ds,batch_size=2,shuffle=False,num_workers=2)

class ConvBlock(nn.Module):
    def __init__(self,a,b):
        super().__init__(); self.block=nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(True),nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(True))
    def forward(self,x): return self.block(x)
class DualStemUNet(nn.Module):
    def __init__(self):
        super().__init__(); f=models.mobilenet_v3_small(weights=None).features
        self.sar_stem=nn.Sequential(nn.Conv2d(3,16,3,2,1,bias=False),nn.BatchNorm2d(16),nn.Hardswish(True))
        self.opt_stem=nn.Sequential(nn.Conv2d(4,16,3,2,1,bias=False),nn.BatchNorm2d(16),nn.Hardswish(True))
        self.enc1=f[1]; self.enc2=f[2:4]; self.enc3=f[4:9]; self.enc4=f[9:12]
        self.up4=nn.ConvTranspose2d(96,48,2,2); self.dec4=ConvBlock(96,48); self.up3=nn.ConvTranspose2d(48,24,2,2); self.dec3=ConvBlock(48,24)
        self.up2=nn.ConvTranspose2d(24,16,2,2); self.dec2=ConvBlock(32,16); self.up1=nn.ConvTranspose2d(16,16,2,2); self.dec1=ConvBlock(32,16)
        self.up0=nn.ConvTranspose2d(16,16,2,2); self.dec0=ConvBlock(16,16); self.head=nn.Conv2d(16,2,1)
    def forward(self,x,opt_gate=None):
        opt=x[:,OPT]; base=x[:,SAR+DEM]
        if opt_gate is None: opt_gate=(opt.abs().sum((1,2,3))>0).float()
        e0=self.sar_stem(base)+opt_gate.view(-1,1,1,1)*self.opt_stem(opt)
        e1=self.enc1(e0); e2=self.enc2(e1); e3=self.enc3(e2); e4=self.enc4(e3)
        d4=self.dec4(torch.cat([self.up4(e4),e3],1)); d3=self.dec3(torch.cat([self.up3(d4),e2],1)); d2=self.dec2(torch.cat([self.up2(d3),e1],1)); d1=self.dec1(torch.cat([self.up1(d2),e0],1))
        return self.head(self.dec0(self.up0(d1)))

def score(model,loader,zero_opt=False,device=DEVICE):
    model.eval(); tp=fp=fn=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device); y=y.to(device)
            if zero_opt: x=x.clone(); x[:,OPT]=0
            pred=model(x).argmax(1); valid=y!=255; p=(pred==1)&valid; t=(y==1)&valid
            tp+=(p&t).sum().item(); fp+=(p&~t).sum().item(); fn+=(~p&t).sum().item()
    return tp/(tp+fp+fn) if tp+fp+fn else 0.

SEEDS=[42,43,44,45,46]; CKPT_DIR=os.path.join(WORK_DIR,'five_fp32'); os.makedirs(CKPT_DIR,exist_ok=True)
ckpts={}; val_history=[]
for seed in SEEDS:
    print('Training seed',seed); seed_everything(seed)
    tr=DataLoader(train_ds,batch_size=4,shuffle=True,num_workers=2,generator=torch.Generator().manual_seed(seed))
    model=DualStemUNet().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=25,eta_min=1e-5); loss_fn=nn.CrossEntropyLoss(ignore_index=255)
    best=-1.; path=os.path.join(CKPT_DIR,f'fp32_seed_{seed}.pth')
    for epoch in range(25):
        model.train(); total=0.
        for x,y in tr:
            x=x.to(DEVICE); y=y.to(DEVICE); drop=torch.rand(x.size(0),device=DEVICE)<.5
            for bi in torch.where(drop)[0].tolist(): x[bi,OPT]=0.
            opt.zero_grad(set_to_none=True); loss=loss_fn(model(x,opt_gate=(~drop).float()),y); loss.backward(); opt.step(); total+=loss.item()
        sch.step(); vf=score(model,valid_loader); vo=score(model,valid_loader,True); hm=2*vf*vo/(vf+vo+1e-8)
        val_history.append({'seed':seed,'epoch':epoch+1,'loss':total/len(tr),'val_full':vf,'val_optical_ablated':vo,'val_hmean':hm})
        print(f'{epoch+1:02}/25 loss={total/len(tr):.4f} full={vf:.4f} optical-ablated={vo:.4f} hmean={hm:.4f}')
        if hm>best: best=hm; torch.save(model.state_dict(),path)
    ckpts[seed]=path; del model,opt,sch
    if torch.cuda.is_available(): torch.cuda.empty_cache()
pd.DataFrame(val_history).to_csv(os.path.join(WORK_DIR,'validation_history.csv'),index=False)

Training seed 42
01/25 loss=0.3823 full=0.6104 optical-ablated=0.5434 hmean=0.5750
02/25 loss=0.2383 full=0.6190 optical-ablated=0.5531 hmean=0.5842
03/25 loss=0.2007 full=0.6029 optical-ablated=0.5202 hmean=0.5585
04/25 loss=0.1787 full=0.6512 optical-ablated=0.5577 hmean=0.6008
05/25 loss=0.1921 full=0.6574 optical-ablated=0.5556 hmean=0.6022
06/25 loss=0.1803 full=0.5902 optical-ablated=0.4874 hmean=0.5339
07/25 loss=0.1809 full=0.6506 optical-ablated=0.5483 hmean=0.5951
08/25 loss=0.1778 full=0.6686 optical-ablated=0.5681 hmean=0.6143
09/25 loss=0.1649 full=0.6921 optical-ablated=0.5435 hmean=0.6088
10/25 loss=0.1507 full=0.6296 optical-ablated=0.4555 hmean=0.5286
11/25 loss=0.1679 full=0.6738 optical-ablated=0.5689 hmean=0.6169
12/25 loss=0.1564 full=0.6633 optical-ablated=0.5038 hmean=0.5726
13/25 loss=0.1428 full=0.6746 optical-ablated=0.4902 hmean=0.5678
14/25 loss=0.1380 full=0.6562 optical-ablated=0.3400 hmean=0.4480
15/25 loss=0.1456 full=0.7001 optical-ablated=0.5293 hmean=

## Paired FP32 / INT8 PTQ 2×2 evaluation on the same test tiles

In [9]:
import copy,gc
from torch.ao.quantization.quantize_fx import prepare_fx,convert_fx
from torch.ao.quantization import get_default_qconfig_mapping
engine='x86' if 'x86' in torch.backends.quantized.supported_engines else 'fbgemm'
torch.backends.quantized.engine=engine
class Wrapper(nn.Module):
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,x): return self.m(x)
rows=[]; sizes=[]; QDIR=os.path.join(WORK_DIR,'five_int8'); os.makedirs(QDIR,exist_ok=True)
for seed in SEEDS:
    fp=DualStemUNet().cpu().eval(); fp.load_state_dict(torch.load(ckpts[seed],map_location='cpu',weights_only=True))
    for cond,z in [('full',False),('optical_ablated',True)]:
        v=score(fp,test_loader,z,device='cpu'); rows.append({'seed':seed,'precision':'FP32','condition':cond,'water_iou':v}); print(seed,'FP32',cond,v)
    fpath=os.path.join(QDIR,f'fp32_seed_{seed}.pth'); torch.save(fp.state_dict(),fpath); sizes.append({'seed':seed,'precision':'FP32','size_MB':os.path.getsize(fpath)/2**20})
    wrapped=Wrapper(copy.deepcopy(fp)).eval(); example=next(iter(cal_loader))[0][:1].cpu()
    prepared=prepare_fx(wrapped,get_default_qconfig_mapping(engine),(example,))
    # Calibration only on training tiles, both full and optical-zeroed inputs.
    with torch.no_grad():
        for n,(x,_) in enumerate(cal_loader):
            x=x.cpu(); prepared(x); z=x.clone(); z[:,OPT]=0; prepared(z)
            if n==19: break
    q=convert_fx(prepared).eval(); q(example)
    qpath=os.path.join(QDIR,f'int8_seed_{seed}.pt'); torch.save(q,qpath); sizes.append({'seed':seed,'precision':'INT8 PTQ','size_MB':os.path.getsize(qpath)/2**20})
    for cond,z in [('full',False),('optical_ablated',True)]:
        v=score(q,test_loader,z,device='cpu'); rows.append({'seed':seed,'precision':'INT8 PTQ','condition':cond,'water_iou':v}); print(seed,'INT8 PTQ',cond,v)
    del fp,wrapped,prepared,q; gc.collect()
results=pd.DataFrame(rows); results.to_csv(os.path.join(WORK_DIR,'paired_2x2_results.csv'),index=False)
pd.DataFrame(sizes).to_csv(os.path.join(WORK_DIR,'model_sizes.csv'),index=False)
print(results.pivot_table(index=['seed','precision'],columns='condition',values='water_iou'))

42 FP32 full 0.7270217949989884
42 FP32 optical_ablated 0.641487233410357


/tmp/ipykernel_49/3165098087.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared=prepare_fx(wrapped,get_default_qconfig_mapping(engine),(example,))
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated

42 INT8 PTQ full 0.7242632839528076
42 INT8 PTQ optical_ablated 0.6337905410337525
43 FP32 full 0.7333348913866752
43 FP32 optical_ablated 0.6578502223316116


/tmp/ipykernel_49/3165098087.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared=prepare_fx(wrapped,get_default_qconfig_mapping(engine),(example,))
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated

43 INT8 PTQ full 0.7340053247918373
43 INT8 PTQ optical_ablated 0.655912862452176
44 FP32 full 0.7423497084552663
44 FP32 optical_ablated 0.6573810139548457


/tmp/ipykernel_49/3165098087.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared=prepare_fx(wrapped,get_default_qconfig_mapping(engine),(example,))
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated

44 INT8 PTQ full 0.7458972335672098
44 INT8 PTQ optical_ablated 0.6535506494767872
45 FP32 full 0.7368332551324404
45 FP32 optical_ablated 0.6482374781609069


/tmp/ipykernel_49/3165098087.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared=prepare_fx(wrapped,get_default_qconfig_mapping(engine),(example,))
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated

45 INT8 PTQ full 0.7357266247930799
45 INT8 PTQ optical_ablated 0.6335906062210217
46 FP32 full 0.7563717205712752
46 FP32 optical_ablated 0.6192894641543975


/tmp/ipykernel_49/3165098087.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared=prepare_fx(wrapped,get_default_qconfig_mapping(engine),(example,))
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated

46 INT8 PTQ full 0.7555013339857293
46 INT8 PTQ optical_ablated 0.5945152475403771
condition           full  optical_ablated
seed precision                           
42   FP32       0.727022         0.641487
     INT8 PTQ   0.724263         0.633791
43   FP32       0.733335         0.657850
     INT8 PTQ   0.734005         0.655913
44   FP32       0.742350         0.657381
     INT8 PTQ   0.745897         0.653551
45   FP32       0.736833         0.648237
     INT8 PTQ   0.735727         0.633591
46   FP32       0.756372         0.619289
     INT8 PTQ   0.755501         0.594515


## Per-seed interaction and summary

The interaction is the difference between the optical-ablation IoU drop under INT8 and the drop under FP32. Positive/negative direction is descriptive; five seeds alone do not establish statistical significance.

In [10]:
w=results.pivot(index='seed',columns=['precision','condition'],values='water_iou')
for p in ['FP32','INT8 PTQ']:
    for c in ['full','optical_ablated']:
        v=w[(p,c)]; print(p,c,'mean=',v.mean(),'SD=',v.std(ddof=1))
fp_drop=w[('FP32','full')]-w[('FP32','optical_ablated')]
q_drop=w[('INT8 PTQ','full')]-w[('INT8 PTQ','optical_ablated')]
inter=q_drop-fp_drop
paired=pd.DataFrame({'seed':w.index,'fp32_optical_drop':fp_drop,'int8_optical_drop':q_drop,'interaction':inter})
print(paired); print('Interaction mean ± SD:',inter.mean(),inter.std(ddof=1))
paired.to_csv(os.path.join(WORK_DIR,'interaction_by_seed.csv'),index=False)

FP32 full mean= 0.7391822741089291 SD= 0.01110289080663163
FP32 optical_ablated mean= 0.6448490824024237 SD= 0.01582842606941163
INT8 PTQ full mean= 0.7390787602181328 SD= 0.01196523509866522
INT8 PTQ optical_ablated mean= 0.6342719813448229 SD= 0.024603296118990987
      seed  fp32_optical_drop  int8_optical_drop  interaction
seed                                                         
42      42           0.085535           0.090473     0.004938
43      43           0.075485           0.078092     0.002608
44      44           0.084969           0.092347     0.007378
45      45           0.088596           0.102136     0.013540
46      46           0.137082           0.160986     0.023904
Interaction mean ± SD: 0.010473587166804531 0.008542109917887859


In [14]:
import copy
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
from torch.ao.quantization import get_default_qconfig_mapping

seed = 42
engine = "x86" if "x86" in torch.backends.quantized.supported_engines else "fbgemm"
torch.backends.quantized.engine = engine

# Rebuild this seed's FP32 model from its state dictionary.
fp32 = DualStemUNet().cpu().eval()
fp32.load_state_dict(
    torch.load(ckpts[seed], map_location="cpu", weights_only=True)
)

# Re-apply PTQ in memory using training data only, under both input conditions.
wrapper = Wrapper(copy.deepcopy(fp32)).eval()
example = next(iter(cal_loader))[0][:1].cpu()
prepared = prepare_fx(wrapper, get_default_qconfig_mapping(engine), (example,))

with torch.no_grad():
    for i, (x, _) in enumerate(cal_loader):
        x = x.cpu()
        prepared(x)                         # full input
        x_ablated = x.clone()
        x_ablated[:, OPT] = 0
        prepared(x_ablated)                 # optical ablated
        if i == 19:
            break

int8 = convert_fx(prepared).eval()
print("FP32 and INT8 models ready for CPU timing.")

/tmp/ipykernel_49/1455816046.py:18: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared = prepare_fx(wrapper, get_default_qconfig_mapping(engine), (example,))
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprec

FP32 and INT8 models ready for CPU timing.


In [15]:
import time
import numpy as np
import torch

torch.set_num_threads(1)
x = next(iter(test_loader))[0][:1].cpu()

def benchmark(model, x, ablate_optical=False, warmup=10, runs=50):
    sample = x.clone()
    if ablate_optical:
        sample[:, OPT] = 0

    with torch.inference_mode():
        for _ in range(warmup):
            model(sample)

        times = []
        for _ in range(runs):
            start = time.perf_counter()
            model(sample)
            times.append((time.perf_counter() - start) * 1000)

    return np.mean(times), np.std(times), np.percentile(times, 95)

for name, model in [("FP32", fp32), ("INT8 PTQ", int8)]:
    for ablated in [False, True]:
        mean, sd, p95 = benchmark(model, x, ablate_optical=ablated)
        condition = "optical-ablated" if ablated else "full"
        print(f"{name} {condition}: {mean:.1f} ± {sd:.1f} ms/tile; p95={p95:.1f} ms")

FP32 full: 194.1 ± 4.5 ms/tile; p95=200.2 ms
FP32 optical-ablated: 198.2 ± 5.9 ms/tile; p95=207.0 ms
INT8 PTQ full: 176.8 ± 4.6 ms/tile; p95=181.2 ms
INT8 PTQ optical-ablated: 178.6 ± 6.3 ms/tile; p95=184.5 ms
